# The N-Queen

In [31]:
import sys
import time
from ortools.sat.python import cp_model

In [32]:
model = cp_model.CpModel()
board_size = 4

In [33]:
# There are `board_size` number of variables, one for a queen in each column q[j] = i
# of the board. The value of each variable is the row that the queen is in.
queens = [model.new_int_var(0, board_size - 1, f"x_{i}") for i in range(board_size)]
print(queens)
print(queens[1]+1)

[x_0(0..3), x_1(0..3), x_2(0..3), x_3(0..3)]
(x_1 + 1)


In [34]:
# All rows must be different.
model.add_all_different(queens)

# No two queens can be on the same diagonal.
model.add_all_different(queens[i] + i for i in range(board_size))
model.add_all_different(queens[i] - i for i in range(board_size))

In [35]:
class NQueenSolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, queens: list[cp_model.IntVar]):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__queens = queens
        self.__solution_count = 0
        self.__start_time = time.time()

    @property
    def solution_count(self) -> int:
        return self.__solution_count

    def on_solution_callback(self):
        current_time = time.time()
        print(
            f"Solution {self.__solution_count}, "
            f"time = {current_time - self.__start_time} s"
        )
        self.__solution_count += 1

        all_queens = range(len(self.__queens))
        for i in all_queens:
            for j in all_queens:
                if self.value(self.__queens[j]) == i:
                    # There is a queen in column j, row i.
                    print("Q", end=" ")
                else:
                    print("_", end=" ")
            print()
        print()

In [37]:
solver = cp_model.CpSolver()
# solution_printer = NQueenSolutionPrinter(queens)
# solver.parameters.enumerate_all_solutions = True
# solver.solve(model, solution_printer)
status = solver.solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"成功找到 {board_size} 皇后问题的一个解！\n")
        
    # 打印可视化棋盘
    for i in range(board_size):
        # 获取第 i 行皇后所在的列号
        queen_col = solver.Value(queens[i])
        
        # 绘制当前行
        row_str = ""
        for j in range(board_size):
            if j == queen_col:
                row_str += " ♛ "  # 皇后的位置
            else:
                row_str += " . "   # 空白位置
        print(row_str)
else:
    print("没有找到解决方案。")

成功找到 4 皇后问题的一个解！

 .  .  ♛  . 
 ♛  .  .  . 
 .  .  .  ♛ 
 .  ♛  .  . 


# Setting a time limit for the solver

In [42]:
"""Solves a problem with a time limit."""

from ortools.sat.python import cp_model


def solve_with_time_limit_sample_sat():
    """Minimal CP-SAT example to showcase calling the solver."""
    # Creates the model.
    model = cp_model.CpModel()
    # Creates the variables.
    num_vals = 3
    x = model.new_int_var(0, num_vals - 1, "x")
    y = model.new_int_var(0, num_vals - 1, "y")
    z = model.new_int_var(0, num_vals - 1, "z")
    # Adds an all-different constraint.
    model.add(x != y)

    # Creates a solver and solves the model.
    solver = cp_model.CpSolver()

    # Sets a time limit of 10 seconds.
    solver.parameters.max_time_in_seconds = 10.0

    status = solver.solve(model)

    if status == cp_model.FEASIBLE:
        print(f"x = {solver.value(x)}")
        print(f"y = {solver.value(y)}")
        print(f"z = {solver.value(z)}")


solve_with_time_limit_sample_sat()

# The Job Shop Problem
![车间作业调度问题（JSSP）](./pics/schedule1.png)  

In [1]:
import collections
from ortools.sat.python import cp_model

In [ ]:
# 1. 业务数据定义
# jobs[i] 表示第 i 个工件。内部列表是按先后顺序执行的工序：(机器ID, 耗时)
jobs_data = [
    [(0, 3), (1, 2), (2, 2)],  # 工件 0: 先在机器0做3小时，再去机器1做2小时，再去机器2做2小时
    [(0, 2), (2, 1), (1, 4)],  # 工件 1: 先在机器0做2小时，再去机器2做1小时，再去机器1做4小时
    [(1, 4), (2, 3)]           # 工件 2: 先在机器1做4小时，再去机器2做3小时 (这个工件只有2道工序)
]

num_machines = 3

# 计算一个绝对安全的最大时间界限 (Horizon)
# 最坏的情况就是所有任务串行执行，总时间不会超过所有耗时之和
horizon = sum(task[1] for job in jobs_data for task in job)

In [44]:
# 2. 创建模型
model = cp_model.CpModel()

In [45]:
# 存储变量的数据字典
# all_tasks[job_id, task_id] = { 'start': 变量, 'end': 变量, 'interval': 区间变量 }
all_tasks = {}

# machine_to_intervals[machine_id] = [区间变量1, 区间变量2, ...]
machine_to_intervals = collections.defaultdict(list)

# 3. 创建变量
for job_id, job in enumerate(jobs_data):
    for task_id, task in enumerate(job):
        machine_id = task[0]
        duration = task[1]
        suffix = f'_j{job_id}_t{task_id}_m{machine_id}'

        # 定义开始时间和结束时间变量 (范围从 0 到 horizon)
        start_var = model.new_int_var(0, horizon, 'start' + suffix)
        end_var = model.new_int_var(0, horizon, 'end' + suffix)
        
        # 🌟 核心：定义区间变量 (Interval Variable)
        interval_var = model.new_interval_var(
            start_var, duration, end_var, 'interval' + suffix)
        
        all_tasks[job_id, task_id] = {
            'start': start_var,
            'end': end_var,
            'interval': interval_var
        }
        # 把区间变量记录到对应的机器下
        machine_to_intervals[machine_id].append(interval_var)

print(all_tasks)
print(machine_to_intervals)

{(0, 0): {'start': start_j0_t0_m0(0..21), 'end': end_j0_t0_m0(0..21), 'interval': interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0)}, (0, 1): {'start': start_j0_t1_m1(0..21), 'end': end_j0_t1_m1(0..21), 'interval': interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1)}, (0, 2): {'start': start_j0_t2_m2(0..21), 'end': end_j0_t2_m2(0..21), 'interval': interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2)}, (1, 0): {'start': start_j1_t0_m0(0..21), 'end': end_j1_t0_m0(0..21), 'interval': interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)}, (1, 1): {'start': start_j1_t1_m2(0..21), 'end': end_j1_t1_m2(0..21), 'interval': interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2)}, (1, 2): {'start': start_j1_t2_m1(0..21), 'end': end_j1_t2_m1(0..21), 'interval': interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1)}, (2, 0): {'start': start_j2_t0_m1(0..21), 'end': end_j2_t0_m1(0..21), 

```python
# all_tasks
{
    (0, 0): {'start': start_j0_t0_m0(0..21), 'end': end_j0_t0_m0(0..21), 'interval': interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0)}, 
    (0, 1): {'start': start_j0_t1_m1(0..21), 'end': end_j0_t1_m1(0..21), 'interval': interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1)}, 
    (0, 2): {'start': start_j0_t2_m2(0..21), 'end': end_j0_t2_m2(0..21), 'interval': interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2)}, 
    (1, 0): {'start': start_j1_t0_m0(0..21), 'end': end_j1_t0_m0(0..21), 'interval': interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)}, 
    (1, 1): {'start': start_j1_t1_m2(0..21), 'end': end_j1_t1_m2(0..21), 'interval': interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2)}, 
    (1, 2): {'start': start_j1_t2_m1(0..21), 'end': end_j1_t2_m1(0..21), 'interval': interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1)}, 
    (2, 0): {'start': start_j2_t0_m1(0..21), 'end': end_j2_t0_m1(0..21), 'interval': interval_j2_t0_m1(start = start_j2_t0_m1, size = 4, end = end_j2_t0_m1)}, 
    (2, 1): {'start': start_j2_t1_m2(0..21), 'end': end_j2_t1_m2(0..21), 'interval': interval_j2_t1_m2(start = start_j2_t1_m2, size = 3, end = end_j2_t1_m2)}
}

# machine_to_intervals
defaultdict(<class 'list'>, 
    {
        0: [interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0), interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)], 
        1: [interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1), interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1), interval_j2_t0_m1(start = start_j2_t0_m1, size = 4, end = end_j2_t0_m1)], 
        2: [interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2), interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2), interval_j2_t1_m2(start = start_j2_t1_m2, size = 3, end = end_j2_t1_m2)]
    }
)
```

In [46]:
# 4. 添加约束
# 约束 A：同一个工件的工序必须按顺序执行 (后一个工序的开始时间 >= 前一个工序的结束时间)
for job_id, job in enumerate(jobs_data):
    for task_id in range(1, len(job)):
        prev_end = all_tasks[job_id, task_id - 1]['end']
        curr_start = all_tasks[job_id, task_id]['start']
        model.add(curr_start >= prev_end)

# 约束 B：🌟 核心：同一台机器上的任何任务不能在时间上重叠
for machine_id in range(num_machines):
    model.add_no_overlap(machine_to_intervals[machine_id])

In [47]:
# 5. 定义目标函数：最小化总完工时间 (Makespan)
# 总完工时间 >= 所有工件最后一个工序的结束时间
makespan = model.new_int_var(0, horizon, 'makespan')
makespan_ends = []
for job_id, job in enumerate(jobs_data):
    last_task_id = len(job) - 1
    makespan_ends.append(all_tasks[job_id, last_task_id]['end'])
    
model.add_max_equality(makespan, makespan_ends) # makespan = max(所有最后一个工序的end)
model.minimize(makespan)

In [48]:
# 6. 调用求解器
solver = cp_model.CpSolver()
status = solver.Solve(model)

In [50]:
# 7. 打印结果 (以机器维度格式化输出排产计划)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f'最优总完工时间 (Makespan): {solver.objective_value} 小时\n')

    # 组织输出数据
    assigned_jobs = collections.defaultdict(list)
    for job_id, job in enumerate(jobs_data):
        for task_id, task in enumerate(job):
            machine_id = task[0]
            assigned_jobs[machine_id].append({
                'job': job_id,
                'task': task_id,
                'start': solver.value(all_tasks[job_id, task_id]['start']),
                'duration': task[1]
            })

    # 按机器打印时间表
    for machine_id in range(num_machines):
        # 将该机器上的任务按开始时间排序
        assigned_jobs[machine_id].sort(key=lambda x: x['start'])
        
        schedule_str = f'机器 {machine_id}: '
        for task_dict in assigned_jobs[machine_id]:
            job = task_dict['job']
            task = task_dict['task']
            start = task_dict['start']
            duration = task_dict['duration']
            end = start + duration
            
            schedule_str += f'[工件{job}_工序{task}: {start:>2}点 -> {end:>2}点]  '
        print(schedule_str)
else:
    print('没有找到可行的排产方案。')

最优总完工时间 (Makespan): 11.0 小时

机器 0: [工件1_工序0:  0点 ->  2点]  [工件0_工序0:  2点 ->  5点]  
机器 1: [工件2_工序0:  0点 ->  4点]  [工件0_工序1:  5点 ->  7点]  [工件1_工序2:  7点 -> 11点]  
机器 2: [工件1_工序1:  2点 ->  3点]  [工件2_工序1:  4点 ->  7点]  [工件0_工序2:  7点 ->  9点]  


DAG
负责描述实验步骤依赖
        ↓
RCPSP
把 DAG + 资源约束转成调度优化问题
        ↓
Rolling Horizon Scheduling
每隔一段时间构建一个局部 RCPSP 问题
        ↓
CP-SAT
求解这个局部调度问题，输出可执行计划 -> DAG (给用户输出？)
        ↓
硬件层      


# 直接交换死锁

在实验室自动化（Lab Automation）或多机械臂矩阵中，如果系统控制逻辑是：“机械臂每次运动前，必须判断目标设备（坑位）是否为空”，就会极易引发直接交换死锁 (Direct Exchange Deadlock)。

典型崩溃场景：

- 状态： 孵化器（满，装有板子A），离心机（满，装有板子B）。两者均已完成实验。
- 目标： 板子A需要去离心机，板子B需要去孵化器。
- 死锁原因： 机械臂尝试抓取板子A时，判断目标（离心机）不为空，于是拒绝动作，原地等待；同理，另一侧判断孵化器不为空，也原地等待。
- 本质： 系统试图在没有“中间空地（临时变量）”的情况下，让两个物理实体互换位置。如同两人在狭窄独木桥上相遇，都等对方先退，导致永远僵持。

---

# 破解方案
1. 引入“物理缓存站（Buffer/Hotel）”

在软件工程中，交换两个变量 a 和 b 需要一个临时变量 temp。物理硬件同理。

设计思路： 在工作台面上预留至少一个公共的、空闲的板位（通常称为 Buffer Rack 或 Hotel）。
运行逻辑（中心调度器强行介入）：
- 中央调度器检测到死锁条件成立（双方皆满且互需位置）。
- 拦截常规等待指令，下发**“死锁消解宏指令”**。
- 臂 A 取出孵化器里的板子，但不去离心机，而是放到 Buffer位 上。 (此时：孵化器腾空！)
- 臂 B 看到孵化器空了，取出离心机里的板子，放入孵化器。 (此时：离心机腾空！)
- 臂 A 去 Buffer 位拿回原板子，放入离心机。完成完美交换。
评价： 最安全、最标准、最容易用软件实现的方案(但有空间要求)。

2. 修改软件检测时机（空中暂存法）

如果硬件台面极其紧凑，连一个 Buffer 暂存位都无法增加，则必须打破机械臂“不见空位不提件”的保守逻辑。

设计思路： 允许机械臂在不满足“目标为空”的情况下，先把物料提出来抓在手里。
运行逻辑：
- 调度器强行命令 臂 A 忽略离心机状态，直接把板子从孵化器里提出来，并停留在安全的半空中等待。 (此时：孵化器物理空间被强行释放！)
- 臂 B 按常规逻辑判断：孵化器已空，于是取出离心机的板子放入孵化器。 (离心机空间释放！)
- 悬停在半空的 臂 A 判定离心机已空，顺势降落将板子放入离心机。
⚠️ 硬件风险警告： 此方案必须在软件中划定严苛的 “3D 安全避让区”。必须确保臂 A 在半空悬停时，绝对不会与正在大范围运动的臂 B 发生物理干涉（碰撞）。

3. 华容道算法（N-1 容量限制法则）

如果你玩过经典的“华容道”或“数字拼图”游戏，就知道：要想让方块能移动，盘面上必须始终留有一个空格。

设计思路： 从任务下发的源头（准入控制）杜绝满载。
运行逻辑：
- 假设系统中总共有 N 个能放微孔板的仪器槽位（无额外Buffer）。
- 调度器限制：整个流水线最多只允许同时运行 N−1 块板子。
- 在只有 1个孵化器 + 1个离心机（N=2）的系统中，强制只能同时跑 1 块板子；若要跑 2 块板子，必须接入第 3 台设备（如洗板机）充当天然 Buffer。
评价： 只要系统中永远飘着一个“空位 (Token)”，资源最终都能像拼图一样倒腾开，从数学层面彻底消灭死锁。缺点是牺牲了部分并发效率。

# Simpy模拟

```txt
模拟之前的优化结果
最优总完工时间 (Makespan): 11.0 小时

机器 0: [工件1_工序0:  0点 ->  2点]  [工件0_工序0:  2点 ->  5点]  
机器 1: [工件2_工序0:  0点 ->  4点]  [工件0_工序1:  5点 ->  7点]  [工件1_工序2:  7点 -> 11点]  
机器 2: [工件1_工序1:  2点 ->  3点]  [工件2_工序1:  4点 ->  7点]  [工件0_工序2:  7点 ->  9点]  
```

In [12]:
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

# ─────────────────────────────────────────────
# 自定义异常
# ─────────────────────────────────────────────
class SchedulingInfeasibleError(Exception):
    """CP-SAT 求解器返回 INFEASIBLE / UNKNOWN 时抛出"""

class SchedulingInputError(ValueError):
    """输入数据校验失败时抛出"""

class SchedulingTimeoutNoSolution(Exception):
    """超时且连可行解都没找到（UNKNOWN），需要上层决策"""
    
# 接口数据结构定义
@dataclass
class Task:
    id: str
    duration_ms: int
    required_capability: str        # 如 "ThermalCyclingService"，只描述能力，不绑定物理设备
    earliest_start_ms: int = 0
    deadline_ms: Optional[int] = None   # 修复：必须给默认值，否则继承时报错


@dataclass
class Resource:
    id: str
    capability: str     # 与 Task.required_capability 匹配
    capacity: int = 1   # 通常为 1（互斥资源）


@dataclass
class ScheduleRequest:
    tasks: List[Task]
    # DAG 前置依赖：(A_id, B_id) 表示任务 A 必须在任务 B 开始前完成
    precedence_pairs: List[Tuple[str, str]]
    # 从能力注册表拉取的可用资源列表
    resources: List[Resource]
    # 规划时间窗口（ms），通常 30 分钟 = 1_800_000 ms
    horizon_ms: int
    # 任务优先级权重，紧急样本可设高值；None 时所有任务权重视为 1.0
    priority_weights: Optional[Dict[str, float]] = None


@dataclass
class Assignment:
    task_id: str
    device_id: str        # 具体资源 ID（物理设备在此处才出现）
    planned_start_ms: int
    planned_end_ms: int
    window_slack_ms: int    # 允许战术层在此范围内弹性调整


@dataclass
class ScheduleResult:
    status: str                         # "OPTIMAL" | "FEASIBLE" | "INFEASIBLE"
    solve_time_ms: float                # 实际求解耗时，用于性能监控
    assignments: List[Assignment] = field(default_factory=list)
    makespan_ms: int = 0                # 所有任务完成时刻

In [13]:
jobs_data = [
    [(0, 3), (1, 2), (2, 2)],  # 工件 0: 先在机器0做3小时，再去机器1做2小时，再去机器2做2小时
    [(0, 2), (2, 1), (1, 4)],  # 工件 1: 先在机器0做2小时，再去机器2做1小时，再去机器1做4小时
    [(1, 4), (2, 3)]           # 工件 2: 先在机器1做4小时，再去机器2做3小时 (这个工件只有2道工序)
]

tasks: List[Task]                   = []
precedence_pairs: List[Tuple[str, str]] = []
task_id_counter = 0

for job_idx, operations in enumerate(jobs_data):
    prev_task_id: Optional[str] = None
    for op_idx, (machine_id, duration) in enumerate(operations):
        tid = str(task_id_counter)
        task_id_counter += 1
        tasks.append(Task(
            id                  = tid,
            duration_ms         = duration,
            required_capability = f"machine_{machine_id}",
            earliest_start_ms   = 0,
            deadline_ms         = None,
        ))
        if prev_task_id is not None:
            precedence_pairs.append((prev_task_id, tid))
        prev_task_id = tid

resources = [
    Resource(id=f"res_machine_{i}", capability=f"machine_{i}", capacity=1)
    for i in range(3)
]

jobs = ScheduleRequest(
    tasks            = tasks,
    precedence_pairs = precedence_pairs,
    resources        = resources,
    horizon_ms       = 1800,     
    priority_weights = None,
)

```txt
ScheduleRequest(
    tasks=[
        Task(id='0', duration_ms=3, required_capability='machine_0', earliest_start_ms=0, deadline_ms=None), 
        Task(id='1', duration_ms=2, required_capability='machine_1', earliest_start_ms=0, deadline_ms=None), 
        Task(id='2', duration_ms=2, required_capability='machine_2', earliest_start_ms=0, deadline_ms=None), 
        Task(id='3', duration_ms=2, required_capability='machine_0', earliest_start_ms=0, deadline_ms=None), 
        Task(id='4', duration_ms=1, required_capability='machine_2', earliest_start_ms=0, deadline_ms=None), 
        Task(id='5', duration_ms=4, required_capability='machine_1', earliest_start_ms=0, deadline_ms=None), 
        Task(id='6', duration_ms=4, required_capability='machine_1', earliest_start_ms=0, deadline_ms=None), 
        Task(id='7', duration_ms=3, required_capability='machine_2', earliest_start_ms=0, deadline_ms=None)
        ], 
    precedence_pairs=[('0', '1'), ('1', '2'), ('3', '4'), ('4', '5'), ('6', '7')], 
    resources=[
        Resource(id='res_machine_0', capability='machine_0', capacity=1), 
        Resource(id='res_machine_1', capability='machine_1', capacity=1), 
        Resource(id='res_machine_2', capability='machine_2', capacity=1)
        ], 
    horizon_ms=100, 
    priority_weights=None
    )
```

In [14]:
import time
from ortools.sat.python import cp_model
 
#  超时场景决策树
# ──────────────────────────────────────────────────────
# solver.solve() 返回
#     │
#     ├─ OPTIMAL / FEASIBLE ──→ 正常返回 + 更新缓存 ✅
#     │
#     ├─ UNKNOWN（超时）
#     │       │
#     │       ├─ 有缓存 ──→ 返回 CACHED 解，零额外等待 ✅
#     │       │
#     │       └─ 无缓存 ──→ 应急松弛求解（0.2s 预算）
#     │                   │
#     │                   ├─ 成功 ──→ 返回 EMERGENCY 解 ✅
#     │                   └─ 失败 ──→ 抛 SchedulingTimeoutNoSolution ✅
#     │
#     └─ INFEASIBLE ──→ 抛 SchedulingInfeasibleError ✅
# ──────────────────────────────────────────────────────

class StrategicScheduler:
    """
    第一层：战略调度器
    - 每 30 秒定时触发，或在第二层检测到偏差超阈值时提前触发
    - 硬性时间预算 500ms，超时接受当前最优可行解（FEASIBLE）
    - 多线程并行搜索，充分利用边缘 IPC 多核

    FEASIBLE 优先策略：
    ┌─────────────────────────────────────────────────────────┐
    │  求解结果       处理方式                                  │
    │  ─────────     ──────────────────────────────────────── │
    │  OPTIMAL      返回最优解，更新缓存                        │
    │  FEASIBLE     返回可行解（超时截断），更新缓存             │
    │  UNKNOWN      ① 有缓存 → 返回上次可行解 + 降级标记       │
    │               ② 无缓存 → 启动应急松弛求解                │
    │  INFEASIBLE   约束真正冲突，抛出异常由上层处理            │
    └─────────────────────────────────────────────────────────┘
    """

    TIME_BUDGET_SECONDS: float = 0.5    # 硬性时间预算（关键参数）
    # 应急松弛求解的时间预算（比正常预算更短，保证系统不阻塞）
    FALLBACK_TIME_BUDGET_SECONDS: float = 0.2

    NUM_SEARCH_WORKERS: int    = 4      # 并行搜索线程数
    MAKESPAN_WEIGHT: int       = 10_000 # Makespan 权重，远大于优先级权重
    PRIORITY_SCALE: int        = 1_000  # 优先级浮点 → 整数放大倍数

    HORIZON_MINUTES     = 30    # 规划窗口
    RESCHEDULE_INTERVAL = 30    # 常规重规划间隔（秒）

    def __init__(self) -> None:
        # ✅ 关键：缓存上一次成功的可行解，用于超时降级
        self._last_feasible_result: Optional[ScheduleResult] = None

    # ── 公共入口 ─────────────────────────────────────────────────
    def solve(self, request: ScheduleRequest) -> ScheduleResult:
        """
        执行战略规划，严格遵守 FEASIBLE 优先原则。

        Returns
        -------
        ScheduleResult
            status 字段含义：
            - "OPTIMAL"   : 最优解
            - "FEASIBLE"  : 时间预算内的可行解
            - "CACHED"    : 超时且无新解，返回上次缓存的可行解
            - "EMERGENCY" : 松弛约束后的应急解

        Raises
        ------
        SchedulingInfeasibleError     : 约束真正冲突，无解
        SchedulingTimeoutNoSolution   : 超时且无任何历史缓存，需上层介入
        """

        model, solver, task_vars, cap_resources = self._build_model(request)

        # ── 主求解 ───────────────────────────────────────────────
        t0     = time.perf_counter()
        status = solver.solve(model)
        elapsed_ms = (time.perf_counter() - t0) * 1_000

        # ── 分支处理 ─────────────────────────────────────────────
        if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            # ✅ 正常路径：更新缓存并返回
            result = self._extract_result(
                solver, task_vars, request,
                cap_resources, status, elapsed_ms,
            )
            self._last_feasible_result = result   # 更新缓存
            return result

        elif status == cp_model.UNKNOWN:
            # ⚠️ 超时路径：优先级 ① 缓存 ② 应急松弛
            return self._handle_timeout(request, elapsed_ms)

        else:
            # ❌ 真正的 INFEASIBLE
            raise SchedulingInfeasibleError(
                "约束冲突，无论给多少时间都无法求解。"
                "请检查 precedence_pairs 是否成环，或 deadline_ms 是否过紧。"
            )
    
    # ── 超时降级处理 ─────────────────────────────────────────────
    def _handle_timeout(
        self,
        request: ScheduleRequest,
        primary_elapsed_ms: float,
    ) -> ScheduleResult:
        """
        超时且未找到可行解时的三级降级策略：
        Level 1：返回上次缓存的可行解（最快，0 ms 额外开销）
        Level 2：松弛软约束后应急求解（有限额外时间）
        Level 3：实在无解，抛出异常通知上层
        """

        # Level 1：有缓存，直接返回，标记为 CACHED ─────────────
        if self._last_feasible_result is not None:
            # 返回缓存副本并标记，上层可据此判断是否需要人工介入
            cached = ScheduleResult(
                status        = "CACHED",
                solve_time_ms = primary_elapsed_ms,
                assignments   = self._last_feasible_result.assignments,
                makespan_ms   = self._last_feasible_result.makespan_ms,
            )
            return cached
        
        # Level 2：无缓存，尝试松弛约束应急求解 ──────────────────
        emergency_result = self._emergency_solve(request, primary_elapsed_ms)
        if emergency_result is not None:
            self._last_feasible_result = emergency_result  # 更新缓存
            return emergency_result

        # Level 3：彻底失败，通知上层 ────────────────────────────
        raise SchedulingTimeoutNoSolution(
            f"主求解超时（{primary_elapsed_ms:.1f} ms）且应急求解也失败，"
            "系统无法在时间预算内生成任何可行调度计划。"
            "建议：放宽 deadline_ms 约束 / 减少任务数 / 增加资源。"
        )

     # ── 应急松弛求解 ─────────────────────────────────────────────
    def _emergency_solve(
        self,
        request: ScheduleRequest,
        primary_elapsed_ms: float,
    ) -> Optional[ScheduleResult]:
        """
        松弛策略：
        1. 移除所有 deadline_ms 软约束（仅保留 earliest_start 和 precedence）
        2. 只追求 FEASIBLE，不优化 makespan
        3. 使用更短的时间预算
        """

        # 构造松弛版请求（去掉所有 deadline）
        relaxed_tasks = [
            Task(
                id                  = t.id,
                duration_ms         = t.duration_ms,
                required_capability = t.required_capability,
                earliest_start_ms   = t.earliest_start_ms,
                deadline_ms         = None,   # ✅ 松弛 deadline
            )
            for t in request.tasks
        ]

        relaxed_request = ScheduleRequest(
            tasks            = relaxed_tasks,
            precedence_pairs = request.precedence_pairs,
            resources        = request.resources,
            horizon_ms       = request.horizon_ms,
            priority_weights = None,   # ✅ 松弛优先级目标（纯可行性求解）
        )

        model, solver, task_vars, cap_resources = self._build_model(
            relaxed_request,
            time_budget=self.FALLBACK_TIME_BUDGET_SECONDS,
            feasibility_only=True,   # ✅ 不设优化目标，只求可行解
        )

        t0     = time.perf_counter()
        status = solver.solve(model)
        elapsed_ms = primary_elapsed_ms + (time.perf_counter() - t0) * 1_000

        if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            print("应急松弛求解成功（总耗时=%.1f ms）", elapsed_ms)
            return self._extract_result(
                solver, task_vars, relaxed_request,
                cap_resources, status, elapsed_ms,
                override_status="EMERGENCY",
            )

        print("应急松弛求解也失败：status=%s", solver.status_name(status))
        return None

    # ── 模型构建（抽取为独立方法，供主求解和应急求解复用）────────
    def _build_model(
        self,
        request: ScheduleRequest,
        time_budget: float = None,
        feasibility_only: bool = False,
    ) -> Tuple:
        """
        构建 CP-SAT 模型，返回 (model, solver, task_vars, cap_resources)
        """
        model  = cp_model.CpModel()
        solver = cp_model.CpSolver()

        # ── 硬性时间预算 ─────────────────────────────────
        budget = time_budget if time_budget is not None else self.TIME_BUDGET_SECONDS
        solver.parameters.max_time_in_seconds = budget

        solver.parameters.num_search_workers  = self.NUM_SEARCH_WORKERS  

        horizon   = request.horizon_ms

        # ── 步骤 1：预计算资源容量表 ───────────────────────
        # capability -> 该类型资源总容量（同能力设备可并行）
        cap_capacity: Dict[str, int] = {}

        # capability -> 该类型所有资源 id 列表（用于分配阶段）
        cap_resources: Dict[str, List[str]] = {}

        for res in request.resources:
            cap_capacity[res.capability]  = cap_capacity.get(res.capability, 0) + res.capacity
            cap_resources.setdefault(res.capability, []).append(res.id)

        # ── 步骤 2：为每个任务创建区间变量 ──────────────────────
        # task_id -> (start_var, end_var, interval_var)
        task_vars: Dict[str, Tuple] = {}

        for task in request.tasks:
            lo = task.earliest_start_ms
            hi = task.deadline_ms if task.deadline_ms is not None else horizon

            start    = model.new_int_var(lo, hi,      f"s_{task.id}")
            end      = model.new_int_var(lo, horizon, f"e_{task.id}")
            interval = model.new_interval_var(start, task.duration_ms, end, f"i_{task.id}")

            task_vars[task.id] = (start, end, interval)

            if task.deadline_ms is not None:
                model.add(end <= task.deadline_ms)

        # ── 步骤 3：注入 DAG 前置依赖约束（DAG 引擎接口核心）────
        for pred_id, succ_id in request.precedence_pairs:
            _, pred_end, _ = task_vars[pred_id]
            succ_start, _, _ = task_vars[succ_id]
            model.add(succ_start >= pred_end)

        # ── 步骤 4：资源互斥约束 ─────────────────────────────────
        # 按能力类型分组，同类设备总数即为资源容量
        cap_intervals: Dict[str, List] = {}
        
        for task in request.tasks:
            cap = task.required_capability
            _, _, interval = task_vars[task.id]
            cap_intervals.setdefault(cap, []).append(interval)

        for cap, intervals in cap_intervals.items():
            capacity = cap_capacity.get(cap, 1)
            demands  = [1] * len(intervals)
            model.add_cumulative(intervals, demands, capacity)

        # ── 步骤 5：优化目标（可行性模式下跳过） ───────────────────────────────
        if not feasibility_only:
            makespan = model.new_int_var(0, horizon, "makespan")
            model.add_max_equality(makespan, [e for _, e, _ in task_vars.values()])
            weights = request.priority_weights or {}
            priority_cost = sum(
                int(weights.get(t.id, 1.0) * self.PRIORITY_SCALE) * task_vars[t.id][0]
                for t in request.tasks
            )
            # Makespan 权重 10000 远大于优先级权重，优先保证总时间最短
            model.minimize(makespan * self.MAKESPAN_WEIGHT + priority_cost)

        return model, solver, task_vars, cap_resources

    # ── 私有方法 ─────────────────────────────────────────────────
    def _extract_result(
        self,
        solver: cp_model.CpSolver,
        task_vars: Dict,
        request: ScheduleRequest,
        cap_resources: Dict[str, List[str]],
        status: int,
        elapsed_ms: float,
        override_status: Optional[str] = None,
    ) -> ScheduleResult:
        """从求解器提取结果，构造 ScheduleResult。"""

        if override_status:
            status_str = override_status
        else:
            status_str = "OPTIMAL" if status == cp_model.OPTIMAL else "FEASIBLE"

        assignments: List[Assignment] = []

        # 同一能力组内已分配的资源索引（轮询分配，保证负载均衡）
        cap_alloc_idx: Dict[str, int] = {}

        for task in request.tasks:
            start_var, end_var, _ = task_vars[task.id]
            s = solver.value(start_var)
            e = solver.value(end_var)

            # 资源分配：在同能力组内轮询
            cap  = task.required_capability
            res_list = cap_resources.get(cap, [cap])   # 兜底用 cap 本身
            idx  = cap_alloc_idx.get(cap, 0)
            resource_id = res_list[idx % len(res_list)]
            cap_alloc_idx[cap] = idx + 1

            # window_slack：deadline 与实际结束之差，无 deadline 则取到 horizon
            deadline    = task.deadline_ms if task.deadline_ms is not None else request.horizon_ms
            window_slack = max(0, deadline - e)

            assignments.append(Assignment(
                task_id          = task.id,
                device_id        = resource_id,
                planned_start_ms = s,
                planned_end_ms   = e,
                window_slack_ms  = window_slack,
            ))

        makespan_ms = max(a.planned_end_ms for a in assignments) if assignments else 0

        return ScheduleResult(
            status       = status_str,
            solve_time_ms= elapsed_ms,
            assignments  = assignments,
            makespan_ms  = makespan_ms,
        )

In [ ]:
def print_result(result: ScheduleResult) -> None:
    """格式化打印调度结果（甘特图风格）。"""
    print(f"\n{'─'*55}")
    print(f"  求解状态  : {result.status}")
    print(f"  求解耗时  : {result.solve_time_ms:.2f} ms")
    print(f"  总 Makespan: {result.makespan_ms} ms")
    print(f"{'─'*55}")
    print(f"  {'任务':>4}  {'资源':<18} {'开始':>6} {'结束':>6} {'裕量':>6}")
    print(f"{'─'*55}")
    for a in sorted(result.assignments, key=lambda x: (x.planned_start_ms, x.task_id)):
        print(
            f"  Task {a.task_id:>2}  {a.device_id:<18} "
            f"{a.planned_start_ms:>6}  {a.planned_end_ms:>6}  {a.window_slack_ms:>6}"
        )
    print(f"{'─'*55}\n")


scheduler = StrategicScheduler()
result = scheduler.solve(jobs)
print_result(result)


───────────────────────────────────────────────────────
  求解状态  : OPTIMAL
  求解耗时  : 164.25 ms
  总 Makespan: 11 ms
───────────────────────────────────────────────────────
    任务  资源                     开始     结束     裕量
───────────────────────────────────────────────────────
  Task  3  res_machine_0           0       2    1798
  Task  6  res_machine_1           0       4    1796
  Task  0  res_machine_0           2       5    1795
  Task  4  res_machine_2           2       3    1797
  Task  7  res_machine_2           4       7    1793
  Task  1  res_machine_1           5       7    1793
  Task  2  res_machine_2           7       9    1791
  Task  5  res_machine_1           7      11    1789
───────────────────────────────────────────────────────



In [ ]:
# 战术调度

# 战术层职责边界
# ──────────────────────────────────────────────────────────────
# 事件驱动（毫秒级）          战略层接口
#     │                           │
#     ├─ on_task_ready()          ├─ request_reschedule()   偏差超阈值
#     ├─ on_task_completed()      └─ update_plan()          接收新计划
#     ├─ on_device_fault()
#     └─ on_device_recovered()    注册表接口
#                                 ├─ get_available()        查询可用设备
#                                 └─ get_backup()           查询备用设备
# ──────────────────────────────────────────────────────────────

class TacticalDispatcher:
    """
    第二层：战术调度器
    - 毫秒级响应，处理实时动态事件
    - 在战略计划的时间窗口约束内做局部贪心（First-fit）决策
    - 偏差超阈值时通知战略层提前重规划
    """

    DRIFT_THRESHOLD_MS = 5000   # 偏差超过 5 秒触发重规划

    def on_task_ready(self, task: Task) -> Assignment:
        """前置条件满足，立即分配具体设备（First-fit 贪心）"""
        planned_window = self.current_plan.get_window(task.id)
        candidates     = self.registry.get_available(task.required_capability)

        for device in candidates:
            if device.is_idle() and self._fits_window(device, task, planned_window):
                device.reserve(task)
                return Assignment(task_id=task.id, device_id=device.id)

        # 所有候选设备都忙，等待最早空闲的一台
        earliest = min(candidates, key=lambda d: d.available_at_ms)
        return Assignment(
            task_id=task.id,
            device_id=earliest.id,
            planned_start_ms=earliest.available_at_ms
        )

    def on_task_completed(self, task_id: str, actual_end_ms: int):
        """任务完成回调，检测偏差，决定是否触发提前重规划"""
        planned_end = self.current_plan.get_end(task_id)
        drift = abs(actual_end_ms - planned_end)
        if drift > self.DRIFT_THRESHOLD_MS:
            self.strategic.request_reschedule(
                reason=f"task {task_id} drifted {drift}ms from plan"
            )

    def on_device_fault(self, device_id: str):
        """设备故障，将未执行任务迁移到同能力备用设备"""
        for task in self.current_plan.get_pending_tasks_on(device_id):
            backup = self.registry.get_backup(
                capability=task.required_capability,
                exclude=device_id
            )
            if backup:
                self.current_plan.reassign(task.id, backup.id)
            else:
                # 无备用设备，上报 L4 请求人工干预
                self.emit_alert(
                    f"无同能力备用设备可承接 {task.required_capability}，受影响任务：{task.id}"
                )


In [1]:
import simpy
from dataclasses import dataclass


@dataclass
class Operation:
    job_id: int
    op_id: int
    machine_id: int
    start: float
    end: float

    @property
    def duration(self):
        return self.end - self.start


# 按“工件”组织排程数据
jobs = {
    0: [
        Operation(job_id=0, op_id=0, machine_id=0, start=2, end=5),
        Operation(job_id=0, op_id=1, machine_id=1, start=5, end=7),
        Operation(job_id=0, op_id=2, machine_id=2, start=7, end=9),
    ],
    1: [
        Operation(job_id=1, op_id=0, machine_id=0, start=0, end=2),
        Operation(job_id=1, op_id=1, machine_id=2, start=2, end=3),
        Operation(job_id=1, op_id=2, machine_id=1, start=7, end=11),
    ],
    2: [
        Operation(job_id=2, op_id=0, machine_id=1, start=0, end=4),
        Operation(job_id=2, op_id=1, machine_id=2, start=4, end=7),
    ],
}


def job_process(env, job_id, operations, machines, event_log):
    """
    一个工件的仿真过程。
    工件会按照给定的排程依次执行每道工序。
    """

    for op in operations:
        # 等到计划开始时间
        if env.now < op.start:
            yield env.timeout(op.start - env.now)

        print(
            f"[{env.now:>5.1f}h] 工件{job_id}_工序{op.op_id} "
            f"请求机器{op.machine_id}"
        )

        # 请求对应机器
        with machines[op.machine_id].request() as req:
            yield req

            actual_start = env.now

            # 如果实际开始时间晚于计划开始时间，说明发生了等待或冲突
            if actual_start > op.start:
                print(
                    f"  警告：工件{job_id}_工序{op.op_id} "
                    f"计划 {op.start}h 开始，实际 {actual_start}h 开始"
                )

            print(
                f"[{actual_start:>5.1f}h] 工件{job_id}_工序{op.op_id} "
                f"开始加工，机器{op.machine_id}，预计耗时 {op.duration}h"
            )

            # 加工
            yield env.timeout(op.duration)

            actual_end = env.now

            print(
                f"[{actual_end:>5.1f}h] 工件{job_id}_工序{op.op_id} "
                f"完成，释放机器{op.machine_id}"
            )

            event_log.append({
                "job_id": job_id,
                "op_id": op.op_id,
                "machine_id": op.machine_id,
                "planned_start": op.start,
                "planned_end": op.end,
                "actual_start": actual_start,
                "actual_end": actual_end,
            })


def main():
    env = simpy.Environment()

    # 每台机器容量为 1，表示同一时间只能加工一个工序
    machines = {
        0: simpy.Resource(env, capacity=1),
        1: simpy.Resource(env, capacity=1),
        2: simpy.Resource(env, capacity=1),
    }

    event_log = []

    # 为每个工件创建一个 SimPy 进程
    for job_id, operations in jobs.items():
        env.process(job_process(env, job_id, operations, machines, event_log))

    # 运行仿真
    env.run()

    # 统计 Makespan
    makespan = max(event["actual_end"] for event in event_log)

    print("\n========== 仿真结果 ==========")
    print(f"实际 Makespan: {makespan:.1f} 小时")

    print("\n按机器输出实际执行情况：")
    for machine_id in sorted(machines.keys()):
        machine_events = [
            e for e in event_log if e["machine_id"] == machine_id
        ]
        machine_events.sort(key=lambda x: x["actual_start"])

        print(f"\n机器 {machine_id}:")
        for e in machine_events:
            print(
                f"  工件{e['job_id']}_工序{e['op_id']}: "
                f"{e['actual_start']:.1f} -> {e['actual_end']:.1f}"
            )


if __name__ == "__main__":
    main()


[  0.0h] 工件1_工序0 请求机器0
[  0.0h] 工件2_工序0 请求机器1
[  0.0h] 工件1_工序0 开始加工，机器0，预计耗时 2h
[  0.0h] 工件2_工序0 开始加工，机器1，预计耗时 4h
[  2.0h] 工件0_工序0 请求机器0
[  2.0h] 工件1_工序0 完成，释放机器0
[  2.0h] 工件1_工序1 请求机器2
[  2.0h] 工件1_工序1 开始加工，机器2，预计耗时 1h
[  2.0h] 工件0_工序0 开始加工，机器0，预计耗时 3h
[  3.0h] 工件1_工序1 完成，释放机器2
[  4.0h] 工件2_工序0 完成，释放机器1
[  4.0h] 工件2_工序1 请求机器2
[  4.0h] 工件2_工序1 开始加工，机器2，预计耗时 3h
[  5.0h] 工件0_工序0 完成，释放机器0
[  5.0h] 工件0_工序1 请求机器1
[  5.0h] 工件0_工序1 开始加工，机器1，预计耗时 2h
[  7.0h] 工件1_工序2 请求机器1
[  7.0h] 工件2_工序1 完成，释放机器2
[  7.0h] 工件0_工序1 完成，释放机器1
[  7.0h] 工件0_工序2 请求机器2
[  7.0h] 工件0_工序2 开始加工，机器2，预计耗时 2h
[  7.0h] 工件1_工序2 开始加工，机器1，预计耗时 4h
[  9.0h] 工件0_工序2 完成，释放机器2
[ 11.0h] 工件1_工序2 完成，释放机器1

========== 仿真结果 ==========
实际 Makespan: 11.0 小时

按机器输出实际执行情况：

机器 0:
  工件1_工序0: 0.0 -> 2.0
  工件0_工序0: 2.0 -> 5.0

机器 1:
  工件2_工序0: 0.0 -> 4.0
  工件0_工序1: 5.0 -> 7.0
  工件1_工序2: 7.0 -> 11.0

机器 2:
  工件1_工序1: 2.0 -> 3.0
  工件2_工序1: 4.0 -> 7.0
  工件0_工序2: 7.0 -> 9.0


# 以上是理想情况，现加入随机扰动

| 扰动 | 随机建模方式 |
| :--- | :--- |
| 加工时间波动 | 三角分布 triangular(0.90, 1.25, 1.03) |
| 初始释放延迟 | 85% 无延迟，15% 随机延迟 0.1~0.5 小时 |
| 换型 / 准备延迟 | 80% 无延迟，20% 随机延迟 0.05~0.30 小时 |
| 工序间搬运延迟 | 分段随机，小延迟概率高，大延迟概率低 |
| 机器故障发生时间 | 指数分布，根据平均故障间隔 MTBF 随机生成 |
| 机器维修时间 | 三角分布 triangular(0.15, 1.20, 0.40) |


In [5]:
import simpy
import random
import statistics
from dataclasses import dataclass
from collections import defaultdict


# ======================================================
# 1. 工序数据结构
# ======================================================

@dataclass
class Operation:
    job_id: int
    op_id: int
    machine_id: int
    start: float
    end: float

    @property
    def planned_duration(self):
        return self.end - self.start


# ======================================================
# 2. OR-Tools 输出的理论排程
# ======================================================

jobs = {
    0: [
        Operation(job_id=0, op_id=0, machine_id=0, start=2, end=5),
        Operation(job_id=0, op_id=1, machine_id=1, start=5, end=7),
        Operation(job_id=0, op_id=2, machine_id=2, start=7, end=9),
    ],
    1: [
        Operation(job_id=1, op_id=0, machine_id=0, start=0, end=2),
        Operation(job_id=1, op_id=1, machine_id=2, start=2, end=3),
        Operation(job_id=1, op_id=2, machine_id=1, start=7, end=11),
    ],
    2: [
        Operation(job_id=2, op_id=0, machine_id=1, start=0, end=4),
        Operation(job_id=2, op_id=1, machine_id=2, start=4, end=7),
    ],
}


# ======================================================
# 3. 随机扰动参数
# ======================================================

THEORETICAL_MAKESPAN = 11.0

MACHINE_IDS = [0, 1, 2]

# 每台机器的平均故障间隔，单位：小时
# 数值越小，故障越频繁
MACHINE_MTBF = {
    0: 12.0,
    1: 10.0,
    2: 11.0,
}

# 随机故障生成的时间范围
SIMULATION_HORIZON = 30.0


def sample_process_factor(rng):
    """
    加工时间随机波动系数。

    例如：
    0.95 表示实际加工时间比计划短 5%
    1.10 表示实际加工时间比计划长 10%
    """
    return rng.triangular(0.90, 1.25, 1.03)


def sample_setup_delay(rng):
    """
    换型、装夹、准备、操作员响应延迟。

    80% 概率无额外延迟；
    20% 概率产生 0.05~0.30 小时延迟。
    """
    if rng.random() < 0.80:
        return 0.0
    else:
        return rng.uniform(0.05, 0.30)


def sample_transfer_delay(rng):
    """
    工序间转运、等待物料、搬运延迟。

    大部分情况下延迟较小；
    少数情况下延迟较大。
    """
    p = rng.random()

    if p < 0.70:
        return rng.uniform(0.00, 0.15)
    elif p < 0.95:
        return rng.uniform(0.15, 0.50)
    else:
        return rng.uniform(0.50, 1.00)


def sample_release_delay(rng):
    """
    工件初始释放延迟。
    例如原材料未到、首件确认、工艺文件确认等。
    """
    if rng.random() < 0.85:
        return 0.0
    else:
        return rng.uniform(0.10, 0.50)


def sample_repair_time(rng):
    """
    机器故障后的维修时间。
    """
    return rng.triangular(0.15, 1.20, 0.40)


# ======================================================
# 4. 随机生成机器故障窗口
# ======================================================

def generate_random_machine_downtimes(rng, machine_ids, horizon):
    """
    为每台机器随机生成故障窗口。

    故障发生间隔服从指数分布；
    维修时间服从三角分布。

    返回格式：
    {
        0: [(故障开始, 故障结束), ...],
        1: [(故障开始, 故障结束), ...],
        2: [(故障开始, 故障结束), ...],
    }
    """

    downtimes = {}

    for machine_id in machine_ids:
        mtbf = MACHINE_MTBF[machine_id]
        t = 0.0
        windows = []

        while t < horizon:
            # 下一次故障发生的间隔
            time_to_failure = rng.expovariate(1.0 / mtbf)
            failure_start = t + time_to_failure

            if failure_start >= horizon:
                break

            repair_time = sample_repair_time(rng)
            failure_end = failure_start + repair_time

            windows.append((failure_start, failure_end))

            # 下一轮从维修完成后开始计算
            t = failure_end

        downtimes[machine_id] = windows

    return downtimes


# ======================================================
# 5. 判断机器是否处于故障状态
# ======================================================

def get_current_downtime(env_now, windows):
    """
    如果当前时间处于某个故障窗口内，返回该窗口；
    否则返回 None。
    """
    for start, end in windows:
        if start <= env_now < end:
            return start, end
    return None


def get_next_downtime(env_now, windows):
    """
    获取当前时间之后最近的故障窗口。
    """
    future_windows = [
        (start, end) for start, end in windows
        if start > env_now
    ]

    if not future_windows:
        return None

    return min(future_windows, key=lambda x: x[0])


# ======================================================
# 6. 在机器上工作，考虑随机故障
# ======================================================

def machine_busy_time_with_random_downtime(
    env,
    machine_id,
    busy_time,
    downtimes,
    verbose=True,
    label=""
):
    """
    在机器上执行 busy_time 小时的工作。
    如果过程中遇到机器故障，则暂停，维修完成后继续。

    busy_time 可以代表：
    - 准备时间
    - 加工时间
    """

    remaining = busy_time
    windows = sorted(downtimes.get(machine_id, []))

    while remaining > 1e-9:
        now = env.now

        # 当前是否已经处于故障窗口中
        current_down = get_current_downtime(now, windows)

        if current_down is not None:
            down_start, down_end = current_down

            if verbose:
                print(
                    f"[{env.now:>6.2f}h] 机器{machine_id}当前处于故障中，"
                    f"等待恢复到 {down_end:.2f}h"
                )

            yield env.timeout(down_end - now)
            continue

        # 当前没有故障，寻找下一次故障
        next_down = get_next_downtime(now, windows)

        # 如果后续没有故障，直接完成剩余工作
        if next_down is None:
            yield env.timeout(remaining)
            remaining = 0.0
            break

        down_start, down_end = next_down
        time_until_down = down_start - env.now

        # 如果剩余工作可以在下次故障前完成
        if remaining <= time_until_down:
            yield env.timeout(remaining)
            remaining = 0.0

        else:
            # 先工作到故障发生
            if time_until_down > 0:
                yield env.timeout(time_until_down)
                remaining -= time_until_down

            if verbose:
                print(
                    f"[{env.now:>6.2f}h] 机器{machine_id}随机故障，"
                    f"预计 {down_end:.2f}h 恢复，"
                    f"{label}剩余 {remaining:.2f}h"
                )

            # 等待维修完成
            yield env.timeout(down_end - env.now)

            if verbose:
                print(
                    f"[{env.now:>6.2f}h] 机器{machine_id}维修完成，继续{label}，"
                    f"剩余 {remaining:.2f}h"
                )


# ======================================================
# 7. 工件仿真过程
# ======================================================

def job_process(
    env,
    job_id,
    operations,
    machines,
    event_log,
    downtimes,
    rng,
    verbose=True
):
    """
    一个工件的完整加工过程。
    """

    # 初始释放延迟，随机
    release_delay = sample_release_delay(rng)

    if release_delay > 0:
        if verbose:
            print(
                f"[{env.now:>6.2f}h] 工件{job_id}初始释放延迟 "
                f"{release_delay:.2f}h"
            )
        yield env.timeout(release_delay)

    for index, op in enumerate(operations):

        # 如果当前时间早于理论计划开始时间，则等待到计划开始时间
        if env.now < op.start:
            yield env.timeout(op.start - env.now)

        job_ready_time = env.now

        if verbose:
            if job_ready_time > op.start:
                print(
                    f"[{env.now:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                    f"晚于计划开始时间 {op.start:.2f}h，"
                    f"当前延误 {job_ready_time - op.start:.2f}h"
                )

            print(
                f"[{env.now:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                f"请求机器{op.machine_id}"
            )

        # 按计划开始时间设置优先级
        # 计划越早的工序，优先级越高
        with machines[op.machine_id].request(priority=op.start) as req:
            yield req

            machine_acquired_time = env.now
            machine_wait = machine_acquired_time - job_ready_time

            if verbose and machine_wait > 0:
                print(
                    f"[{env.now:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                    f"等待机器{op.machine_id} {machine_wait:.2f}h"
                )

            # 随机准备 / 换型 / 装夹时间
            setup_delay = sample_setup_delay(rng)

            if setup_delay > 0:
                if verbose:
                    print(
                        f"[{env.now:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                        f"在机器{op.machine_id}上准备/换型，"
                        f"随机耗时 {setup_delay:.2f}h"
                    )

                yield env.process(
                    machine_busy_time_with_random_downtime(
                        env=env,
                        machine_id=op.machine_id,
                        busy_time=setup_delay,
                        downtimes=downtimes,
                        verbose=verbose,
                        label="准备/换型"
                    )
                )

            actual_start = env.now

            # 随机加工时间
            process_factor = sample_process_factor(rng)
            actual_process_time = op.planned_duration * process_factor

            if verbose:
                print(
                    f"[{actual_start:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                    f"开始加工，机器{op.machine_id}，"
                    f"计划耗时 {op.planned_duration:.2f}h，"
                    f"随机系数 {process_factor:.3f}，"
                    f"实际加工净时间 {actual_process_time:.2f}h"
                )

            # 加工过程，考虑随机故障
            yield env.process(
                machine_busy_time_with_random_downtime(
                    env=env,
                    machine_id=op.machine_id,
                    busy_time=actual_process_time,
                    downtimes=downtimes,
                    verbose=verbose,
                    label="加工"
                )
            )

            actual_end = env.now

            if verbose:
                print(
                    f"[{actual_end:>6.2f}h] 工件{job_id}_工序{op.op_id} "
                    f"完成，机器{op.machine_id}释放，"
                    f"计划完成 {op.end:.2f}h，"
                    f"完成延误 {actual_end - op.end:.2f}h"
                )

            event_log.append({
                "job_id": job_id,
                "op_id": op.op_id,
                "machine_id": op.machine_id,

                "planned_start": op.start,
                "planned_end": op.end,
                "planned_duration": op.planned_duration,

                "release_delay": release_delay if index == 0 else 0.0,
                "job_ready_time": job_ready_time,
                "machine_acquired_time": machine_acquired_time,
                "machine_wait": machine_wait,

                "setup_delay": setup_delay,
                "process_factor": process_factor,
                "actual_process_time": actual_process_time,

                "actual_start": actual_start,
                "actual_end": actual_end,

                "start_delay": actual_start - op.start,
                "finish_delay": actual_end - op.end,

                # 机器被该工序占用的总时间，包括准备、故障等待、加工
                "machine_occupied_time": actual_end - machine_acquired_time,
            })

        # 工序间随机搬运 / 等待物料延迟
        if index < len(operations) - 1:
            transfer_delay = sample_transfer_delay(rng)

            if transfer_delay > 0:
                if verbose:
                    print(
                        f"[{env.now:>6.2f}h] 工件{job_id}从工序{op.op_id} "
                        f"转运到下一工序，随机耗时 {transfer_delay:.2f}h"
                    )

                yield env.timeout(transfer_delay)


# ======================================================
# 8. 单次仿真
# ======================================================

def run_once(seed=42, verbose=True):
    rng = random.Random(seed)

    env = simpy.Environment()

    machines = {
        0: simpy.PriorityResource(env, capacity=1),
        1: simpy.PriorityResource(env, capacity=1),
        2: simpy.PriorityResource(env, capacity=1),
    }

    event_log = []

    # 随机生成机器故障窗口
    downtimes = generate_random_machine_downtimes(
        rng=rng,
        machine_ids=MACHINE_IDS,
        horizon=SIMULATION_HORIZON
    )

    if verbose:
        print("\n========== 本次随机生成的机器故障窗口 ==========")
        for machine_id in MACHINE_IDS:
            print(f"机器 {machine_id}:")
            if not downtimes[machine_id]:
                print("  无故障")
            else:
                for start, end in downtimes[machine_id]:
                    print(
                        f"  故障: {start:.2f}h -> {end:.2f}h，"
                        f"维修 {end - start:.2f}h"
                    )

        print("\n========== 开始 SimPy 仿真 ==========\n")

    # 启动每个工件进程
    for job_id, operations in jobs.items():
        env.process(
            job_process(
                env=env,
                job_id=job_id,
                operations=operations,
                machines=machines,
                event_log=event_log,
                downtimes=downtimes,
                rng=rng,
                verbose=verbose
            )
        )

    env.run()

    actual_makespan = max(e["actual_end"] for e in event_log)
    makespan_delay = actual_makespan - THEORETICAL_MAKESPAN

    # 按机器汇总
    machine_events = defaultdict(list)
    for e in event_log:
        machine_events[e["machine_id"]].append(e)

    machine_summary = {}

    for machine_id in MACHINE_IDS:
        events = machine_events[machine_id]

        occupied_time = sum(e["machine_occupied_time"] for e in events)
        productive_time = sum(e["actual_process_time"] for e in events)
        setup_time = sum(e["setup_delay"] for e in events)

        machine_summary[machine_id] = {
            "occupied_time": occupied_time,
            "productive_time": productive_time,
            "setup_time": setup_time,
            "occupied_rate": occupied_time / actual_makespan,
            "productive_rate": productive_time / actual_makespan,
        }

    if verbose:
        print("\n\n========== 单次仿真结果汇总 ==========")
        print(f"理论 Makespan: {THEORETICAL_MAKESPAN:.2f} 小时")
        print(f"实际 Makespan: {actual_makespan:.2f} 小时")
        print(f"Makespan 延误: {makespan_delay:.2f} 小时")

        print("\n========== 各工序实际执行情况 ==========")

        event_log_sorted = sorted(
            event_log,
            key=lambda x: (x["actual_start"], x["machine_id"])
        )

        for e in event_log_sorted:
            print(
                f"工件{e['job_id']}_工序{e['op_id']} | "
                f"机器{e['machine_id']} | "
                f"计划: {e['planned_start']:.2f} -> {e['planned_end']:.2f} | "
                f"实际: {e['actual_start']:.2f} -> {e['actual_end']:.2f} | "
                f"开始延误: {e['start_delay']:.2f}h | "
                f"完成延误: {e['finish_delay']:.2f}h"
            )

        print("\n========== 按机器查看实际排程 ==========")

        for machine_id in MACHINE_IDS:
            print(f"\n机器 {machine_id}:")
            events = sorted(
                machine_events[machine_id],
                key=lambda x: x["actual_start"]
            )

            if not events:
                print("  无任务")
                continue

            for e in events:
                print(
                    f"  工件{e['job_id']}_工序{e['op_id']}: "
                    f"{e['actual_start']:.2f} -> {e['actual_end']:.2f} "
                    f"| 计划 {e['planned_start']:.2f} -> {e['planned_end']:.2f}"
                )

        print("\n========== 机器利用率估算 ==========")

        for machine_id in MACHINE_IDS:
            s = machine_summary[machine_id]
            print(
                f"机器{machine_id}: "
                f"占用时间 {s['occupied_time']:.2f}h，"
                f"占用率 {s['occupied_rate'] * 100:.1f}%；"
                f"净加工时间 {s['productive_time']:.2f}h，"
                f"净加工率 {s['productive_rate'] * 100:.1f}%"
            )

    return {
        "seed": seed,
        "actual_makespan": actual_makespan,
        "makespan_delay": makespan_delay,
        "event_log": event_log,
        "downtimes": downtimes,
        "machine_summary": machine_summary,
    }


# ======================================================
# 9. 多次随机仿真，做统计分析
# ======================================================

def run_many(n=100, base_seed=1000):
    results = []

    for i in range(n):
        seed = base_seed + i
        result = run_once(seed=seed, verbose=False)
        results.append(result)

    makespans = [r["actual_makespan"] for r in results]
    delays = [r["makespan_delay"] for r in results]

    sorted_makespans = sorted(makespans)

    p50 = sorted_makespans[int(0.50 * n)]
    p90 = sorted_makespans[int(0.90 * n)]
    p95 = sorted_makespans[int(0.95 * n) - 1]

    delay_count = sum(1 for x in makespans if x > THEORETICAL_MAKESPAN)

    print("\n========== 多次随机仿真统计 ==========")
    print(f"仿真次数: {n}")
    print(f"理论 Makespan: {THEORETICAL_MAKESPAN:.2f} 小时")
    print(f"平均实际 Makespan: {statistics.mean(makespans):.2f} 小时")
    print(f"最小实际 Makespan: {min(makespans):.2f} 小时")
    print(f"最大实际 Makespan: {max(makespans):.2f} 小时")
    print(f"中位数 P50 Makespan: {p50:.2f} 小时")
    print(f"P90 Makespan: {p90:.2f} 小时")
    print(f"P95 Makespan: {p95:.2f} 小时")
    print(f"平均延误: {statistics.mean(delays):.2f} 小时")
    print(f"超过理论 Makespan 的概率: {delay_count / n * 100:.1f}%")

    print("\n========== 每台机器平均利用情况 ==========")

    for machine_id in MACHINE_IDS:
        occupied_rates = [
            r["machine_summary"][machine_id]["occupied_rate"]
            for r in results
        ]

        productive_rates = [
            r["machine_summary"][machine_id]["productive_rate"]
            for r in results
        ]

        print(
            f"机器{machine_id}: "
            f"平均占用率 {statistics.mean(occupied_rates) * 100:.1f}%；"
            f"平均净加工率 {statistics.mean(productive_rates) * 100:.1f}%"
        )


# ======================================================
# 10. 程序入口
# ======================================================

if __name__ == "__main__":

    # 单次详细仿真
    # run_once(seed=42, verbose=True)

    # 如果想做稳定性评估，可以打开下面这行
    run_many(n=100, base_seed=1000)



========== 多次随机仿真统计 ==========
仿真次数: 100
理论 Makespan: 11.00 小时
平均实际 Makespan: 12.80 小时
最小实际 Makespan: 10.80 小时
最大实际 Makespan: 16.05 小时
中位数 P50 Makespan: 12.78 小时
P90 Makespan: 13.90 小时
P95 Makespan: 14.36 小时
平均延误: 1.80 小时
超过理论 Makespan 的概率: 99.0%

========== 每台机器平均利用情况 ==========
机器0: 平均占用率 44.4%；平均净加工率 41.8%
机器1: 平均占用率 87.9%；平均净加工率 82.9%
机器2: 平均占用率 53.1%；平均净加工率 49.8%
